<a href="https://colab.research.google.com/github/MishterBluesky/TrajMAP-CG/blob/main/TrajMAP_CG_Viewer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# TrajMap- CG

This is a version of trajmap, that simply takes a protein only coarse grain with martini beads (remember to center it first!), and then translates that dependent on the size, to both a pdb file and a heatmap.

Before running, first make sure to use the TM_CG.py script, and run to check usage. the usage accepts .xtc as the trajectory and .gro files as topology files.

Once done this will create a .csv which can be fed to the below.

Feel free to reuse as you see fit.

In [ ]:
!pip install biopython

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio.PDB import PDBParser, PDBIO

# ==========================
# USER INPUTS
# ==========================
csv_file = "12_SpoIIIEmemins_RMSD.csv"   # RMSD CSV
num_chains = 12                        # Number of chains in your system
colormap = "bwr"                   # options: "bwr", "coolwarm", "inferno", "hot"
pdb_file = "SpoIIIEmemonly-poreshape.pdb"             # PDB to assign B-factors
bfactored_pdb = "bfactored_SpoIIIEmemonly-poreshape.pdb"        # Output PDB with B-factors

# ==========================
# LOAD DATA
# ==========================
df = pd.read_csv(csv_file, header=None)

# First row = frame numbers
frames = df.iloc[0, :].values

# Remaining rows = RMSD values (residues x frames)
data = df.iloc[1:, :].values.astype(float)

num_residues, num_frames = data.shape
print(f"Frames: {num_frames}, Residues: {num_residues}")

# ==========================
# CHECK DIVISIBILITY
# ==========================
if num_residues % num_chains != 0:
    raise ValueError("Residues not divisible evenly by number of chains")

residues_per_chain = num_residues // num_chains
print(f"Residues per chain: {residues_per_chain}")

# ==========================
# AVERAGE ACROSS CHAINS PER RESIDUE POSITION
# ==========================
# reshape to (chains, residues_per_chain, frames)
reshaped = data.reshape(num_chains, residues_per_chain, num_frames)

# average across chains (axis 0) → shape: (residues_per_chain, frames)
avg_residues = reshaped.mean(axis=0)

# ==========================
# PLOT HEATMAP
# ==========================
plt.figure(figsize=(14, 8))
sns.heatmap(
    avg_residues,
    cmap=colormap,
    xticklabels=round(num_frames/20),
    vmin=0,      # min color value
    vmax=8       # max color value manually set
)
plt.xlabel("Frame")
plt.ylabel("Residue Position (averaged across chains)")
plt.title("Trajectory RMSD Heatmap (Residue-Averaged Across Chains)")
plt.tight_layout()
plt.show()
# ==========================
# COMPUTE PER-RESIDUE AVERAGE RMSD FOR B-FACTORS
# ==========================
# Average across frames → one value per residue
avg_rmsd_per_residue = avg_residues.mean(axis=1)  # shape: (residues_per_chain,)

print("Average RMSD per residue (Å) calculated for B-factors.")

# ==========================
# ASSIGN TO PDB B-FACTORS
# ==========================
parser = PDBParser(QUIET=True)
structure = parser.get_structure("ref", pdb_file)

# Loop through residues in each chain
for model in structure:
    for chain in model:
        res_list = [res for res in chain if res.id[0] == " "]  # standard residues
        n_assign = min(len(res_list), len(avg_rmsd_per_residue))  # assign only what exists
        for i in range(n_assign):
            bfactor_value = float(avg_rmsd_per_residue[i])
            for atom in res_list[i]:
                atom.set_bfactor(bfactor_value)

# Save new PDB
io = PDBIO()
io.set_structure(structure)
io.save(bfactored_pdb)

print(f"B-factor PDB saved as: {bfactored_pdb}")
print("You can now color by B-factor in PyMOL, e.g., 'spectrum b, red_white_blue'.")
